In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
import joblib

np.random.seed(42)

In [2]:
titanic_raw = pd.read_csv('../data/raw/titanic.csv')
print(titanic_raw.isnull().sum())
titanic_raw.head()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [3]:
def engineer_features(df):
    df = df.copy()
    
    # 1. Family size
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    
    # 2. Is alone
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)
    
    # 3. Title extracted from Name
    df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
    df['Title'] = df['Title'].replace(
        ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 
         'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare'
    )
    df['Title'] = df['Title'].replace(['Mlle', 'Ms'], 'Miss')
    df['Title'] = df['Title'].replace('Mme', 'Mrs')
    
    # 4. Fare per person
    df['FarePerPerson'] = df['Fare'] / df['FamilySize']
    
    # 5. Age bucket (child, adult, senior)
    df['AgeBucket'] = pd.cut(df['Age'], bins=[0, 12, 60, 100], 
                               labels=['Child', 'Adult', 'Senior'])
    
    # 6. Has cabin info (proxy for wealth/location on ship)
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    
    return df


titanic_fe = engineer_features(titanic_raw)
titanic_fe[['FamilySize', 'IsAlone', 'Title', 'FarePerPerson', 'AgeBucket', 'HasCabin']].head()

,FamilySize,IsAlone,Title,FarePerPerson,AgeBucket,HasCabin
0,2,0,Mr,3.62500,Adult,0
1,2,0,Mrs,35.64165,Adult,1
2,1,1,Miss,7.92500,Adult,0
3,2,0,Mrs,26.55000,Adult,1
4,1,1,Mr,8.05000,Adult,0


## Feature Engineering Rationale

1. **FamilySize**: Combines SibSp + Parch — larger families may 
   have had different survival dynamics (harder to coordinate 
   escape, or more help available).

2. **IsAlone**: Solo travellers may behave differently in an 
   emergency than those with family to protect or be protected by.

3. **Title**: Extracted from the passenger's name — captures social 
   status and gender-role information beyond raw Sex (e.g., "Master" 
   indicates a young boy, likely prioritised in evacuation).

4. **FarePerPerson**: Raw Fare is paid per ticket, which can cover 
   multiple family members — dividing by FamilySize gives a more 
   accurate per-individual wealth indicator.

5. **AgeBucket**: Groups continuous age into meaningful categories 
   (Child/Adult/Senior), reflecting the "women and children first" 
   evacuation policy more directly than raw age.

6. **HasCabin**: Whether cabin info is recorded — likely correlates 
   with ticket class and cabin location relative to lifeboats.

In [4]:
# Prepare features and target
X = titanic_fe.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])
y = titanic_fe['Survived']

numeric_features = ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'FarePerPerson']
categorical_features = ['Pclass', 'Sex', 'Embarked', 'Title', 'AgeBucket', 'IsAlone', 'HasCabin']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

full_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42))
])

print(full_pipeline)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'SibSp', 'Parch',
                                                   'Fare', 'FamilySize',
                                                   'FarePerPerson']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                         

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

cv_scores = cross_val_score(full_pipeline, X_train, y_train, cv=5)
print(f"Pipeline CV scores: {cv_scores}")
print(f"Mean: {cv_scores.mean():.4f}, Std: {cv_scores.std():.4f}")

print(f"\nCompare to Day 8 leaked accuracy: 0.XXXX (from your Day 8 notebook)")
print("The pipeline score should be close to, and usually slightly lower than, the leaked version —")
print("because leakage artificially inflated the earlier result.")

Pipeline CV scores: [0.76923077 0.75524476 0.81690141 0.83802817 0.84507042]
Mean: 0.8049, Std: 0.0363

Compare to Day 8 leaked accuracy: 0.XXXX (from your Day 8 notebook)
The pipeline score should be close to, and usually slightly lower than, the leaked version —
because leakage artificially inflated the earlier result.


## Comparing to the Leaked Version (Day 8)

The properly cross-validated pipeline score (~[X]) is close to, 
but slightly lower than, the leaked accuracy from Day 8 (~[X]). 
This is expected: leakage always makes a model look better than 
it truly is, and a correctly built pipeline (where preprocessing 
happens fresh within each fold) gives an honest, generalisable 
estimate of performance.

In [6]:
from sklearn.model_selection import KFold


def target_encode_out_of_fold(X_col, y, n_splits=5, seed=42):
    """Target encode a categorical column using out-of-fold means to avoid leakage."""
    encoded = pd.Series(index=X_col.index, dtype=float)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    global_mean = y.mean()
    
    for train_idx, val_idx in kf.split(X_col):
        train_col = X_col.iloc[train_idx]
        train_y = y.iloc[train_idx]
        val_col = X_col.iloc[val_idx]
        
        category_means = train_y.groupby(train_col).mean()
        encoded.iloc[val_idx] = val_col.map(category_means).fillna(global_mean)
    
    return encoded


# Apply to Title feature as an example
titanic_fe['Title_encoded'] = target_encode_out_of_fold(titanic_fe['Title'], titanic_fe['Survived'])
print(titanic_fe[['Title', 'Survived', 'Title_encoded']].head(10))

    Title  Survived  Title_encoded
0      Mr         0       0.168646
1     Mrs         1       0.783019
2    Miss         1       0.717949
3     Mrs         1       0.775510
4      Mr         0       0.162562
5      Mr         0       0.155131
6      Mr         0       0.151220
7  Master         0       0.580645
8     Mrs         1       0.790476
9     Mrs         1       0.775510


## Why Out-of-Fold Target Encoding Matters

If we simply compute each category's mean target value using the 
*entire* dataset, that mean already "knows" the target for every 
row it's applied to — this is a form of target leakage, since the 
encoding for a row indirectly includes that row's own label.

Using out-of-fold encoding, each row's encoded value is computed 
only from *other* folds it wasn't part of — meaning the encoding 
never has access to its own target value during training, keeping 
the process leakage-free.

In [7]:
full_pipeline.fit(X_train, y_train)
test_score = full_pipeline.score(X_test, y_test)
print(f"Test accuracy: {test_score:.4f}")

joblib.dump(full_pipeline, '../experiments/titanic_pipeline.joblib')
print("Pipeline saved.")

Test accuracy: 0.7933
Pipeline saved.


In [8]:
# Simulate loading in a completely new script
reloaded_pipeline = joblib.load('../experiments/titanic_pipeline.joblib')

# Predict on raw input (a new made-up passenger, engineered same way)
sample_raw = titanic_raw.iloc[[0]].copy()
sample_engineered = engineer_features(sample_raw)
sample_X = sample_engineered.drop(columns=['Survived', 'PassengerId', 'Name', 'Ticket', 'Cabin'])

prediction = reloaded_pipeline.predict(sample_X)
probability = reloaded_pipeline.predict_proba(sample_X)

print(f"Prediction: {prediction[0]} (0=Died, 1=Survived)")
print(f"Probability: {probability[0]}")

Prediction: 0 (0=Died, 1=Survived)
Probability: [0.865 0.135]
